# Culinary RAG Assistant
A Retrieval-Augmented Generation (RAG) system for querying recipe documents. Built for portfolio demonstration using ChromaDB and Gemini API.

In [6]:
from vector_store import VectorStore
from llm_client import LLMClient

store = VectorStore(name="tech_knowledge_base")
llm = LLMClient()

print(f"Database loaded. Total chunks indexed: {store.count()}")

Database loaded. Total chunks indexed: 63


## 1. Test Queries
A test set containing 7 positive queries (answers exist in the database) and 3 negative queries (answers are intentionally missing to test hallucination prevention).

In [ ]:
test_queries = [
    # Пряме запитання (вектор справляється добре)
    "How do you make Peanut Butter Overnight Oats?",
    
    # Складні запитання (граф справляється краще)
    "What equipment do I need if I want to make Peanut Butter Overnight Oats?",
    "Which recipes can I make if I only have a microwave, and no oven?",
    "What are all the ingredients required for Borscht according to our base?",
    "Are there any recipes that use both chia seeds and peanut butter?"
]

## 2. Retrieval & Generation
Execute semantic search for each query (top-3 results), construct the context, and generate grounded answers via LLM.

In [ ]:
from graph_retriever import hybrid_search
import time

with open("rag_test_report.txt", "w", encoding="utf-8") as f:
    for i, query in enumerate(test_queries, 1):
        output_str = f"\n{'='*50}\nQuery {i}: {query}\n{'-'*50}\n"
        print(output_str, end="")
        f.write(output_str)
        
        # Використовуємо гібридний пошук
        full_context = hybrid_search(query, store, top_k=5)
        
        prompt = f"CONTEXT:\n{full_context}\n\nQUESTION: {query}"
        
        try:
            response = llm.chat(prompt=prompt, system=SYSTEM_PROMPT, temperature=0.0)
            # Видалили згадку про sources, оскільки тепер джерела формуються всередині hybrid_search
            answer_str = f"Answer:\n{response.text}\n"
        except Exception as e:
            answer_str = f"Generation Error: {e}\n"
            
        print(answer_str)
        f.write(answer_str)
        
        if i < len(test_queries):
            print("Waiting 15 seconds...")
            time.sleep(15)


Query 1: How do you make Peanut Butter Overnight Oats?
--------------------------------------------------
Answer:
To make Peanut Butter Overnight Oats according to the provided recipe:

**Ingredients:**
* ½ cup unsweetened plain almond milk (or milk of your choice)
* ¾ tbsp. chia seeds
* 2 tbsp. natural peanut butter, almond butter, or nut butter of your choice (creamy or chunky)
* 1 tbsp. maple syrup (or substitute with your preferred sweetener)
* ½ cup rolled oats
* Optional toppings (bananas, strawberries, other fruit, granola, ground flax seeds, or chia seeds)

**Instructions:**
1. Mix all ingredients together.
2. Put the mixture into a storage container with a lid. Make sure the oats are covered by the milk. Store in the refrigerator overnight.
3. Add extra toppings right before eating. It can be eaten cold or warmed in the microwave.

Source: `USU-Student-Cookbook-FINAL-1-12-19.pdf`

Sources retrieved: USU-Student-Cookbook-FINAL-1-12-19.pdf


Testing complete!
